In [ ]:
# =============================
# 1. Install Dependencies
# =============================
!pip install -q transformers accelerate sentence-transformers faiss-cpu pillow matplotlib groq

In [ ]:
# =============================
# 2. Imports
# =============================
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer
from groq import Groq
import faiss
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

In [ ]:
# =============================
# 3. Load Models
# =============================
print("Loading models...")

# BLIP from HuggingFace
caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
caption_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Embedding Model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Groq Client (set your API key before running)
client = Groq(api_key="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")

print("Models loaded.")

In [ ]:
# =============================
# 4. Upload Image
# =============================
from google.colab import files
uploaded = files.upload()

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

plt.imshow(image)
plt.axis('off')

In [ ]:
# =============================
# 5. Generate Image Caption
# =============================
inputs = caption_processor(image, return_tensors="pt")
output = caption_model.generate(**inputs)
caption = caption_processor.decode(output[0], skip_special_tokens=True)

print("Generated Caption:", caption)

In [ ]:
# =============================
# 6. Create Knowledge Base
# =============================
chunks = [caption]
chunk_embeddings = embedder.encode(chunks)

index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings))

In [ ]:
# =============================
# 7. RAG + LLM Generation
# =============================
def query_rag(question):
    # Retrieve context
    q_emb = embedder.encode([question])
    D, I = index.search(np.array(q_emb), k=1)
    context = chunks[I[0][0]]

    # Construct prompt
    prompt = f"""
You are an intelligent vision assistant.

Context from image:
{context}

User question:
{question}

Answer clearly and concisely based only on the image context.
"""

    # Call Groq LLaMA 3.1
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [ ]:
# =============================
# 8. Interactive Query Loop
# =============================
while True:
    q = input("Ask a question about the image (or 'exit'): ")
    if q.lower() == 'exit':
        break
    answer = query_rag(q)
    print("Answer:", answer)